In [2]:
!pip install scikit-learn tqdm

  Using cached scikit_learn-1.8.0-cp313-cp313-win_amd64.whl.metadata (11 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.0 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.0 MB 480.5 kB/s eta 0:00:16
   -- ------------------------------------- 0.5/8.0 MB 480.5 kB/s eta 0:00:16
   -- ------------------------------------- 0.5/8.0 MB 480.5 kB/s eta 0:00:16
   --- ------------------------------------ 0.8/8.0 MB 438.9 kB/s eta 0:00:17
   --- ------------------------------------ 0.8/8.0 MB 438.9 kB/s


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [4]:
LANDMARKS_OUTPUT_DIR = Path("processed_landmarks")

DATASET_CSV = LANDMARKS_OUTPUT_DIR / "video_landmark_dataset.csv"

PREPARED_OUTPUT_DIR = Path("prepared_lstm_data")
PREPARED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
df = pd.read_csv(DATASET_CSV)

print("Total videos:", len(df))
print(df.head())

print("\nClass distribution:")
print(df["label"].value_counts())

Total videos: 113
                                     landmark_path     video_name  label
0  processed_landmarks\deceptive\trial_lie_001.npy  trial_lie_001      1
1  processed_landmarks\deceptive\trial_lie_002.npy  trial_lie_002      1
2  processed_landmarks\deceptive\trial_lie_003.npy  trial_lie_003      1
3  processed_landmarks\deceptive\trial_lie_004.npy  trial_lie_004      1
4  processed_landmarks\deceptive\trial_lie_005.npy  trial_lie_005      1

Class distribution:
label
0    57
1    56
Name: count, dtype: int64


In [6]:
df["file_exists"] = df["landmark_path"].apply(lambda path: Path(path).exists())

print(df["file_exists"].value_counts())

missing_files = df[df["file_exists"] == False]

if len(missing_files) > 0:
    print("Missing files:")
    print(missing_files)
else:
    print("All landmark files exist.")

file_exists
True    113
Name: count, dtype: int64
All landmark files exist.


In [7]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

print("Train videos:", len(train_df))
print("Validation videos:", len(val_df))
print("Test videos:", len(test_df))

print("\nTrain distribution:")
print(train_df["label"].value_counts())

print("\nValidation distribution:")
print(val_df["label"].value_counts())

print("\nTest distribution:")
print(test_df["label"].value_counts())

Train videos: 79
Validation videos: 17
Test videos: 17

Train distribution:
label
0    40
1    39
Name: count, dtype: int64

Validation distribution:
label
1    9
0    8
Name: count, dtype: int64

Test distribution:
label
0    9
1    8
Name: count, dtype: int64


In [8]:
def create_windows_from_sequence(sequence, label, window_size=25, stride=10):
    """
    Converts one video landmark sequence into overlapping windows.

    Args:
        sequence: numpy array of shape (frames, features)
        label: 1 for deceptive, 0 for truthful
        window_size: number of frames per window
        stride: number of frames to move for next window

    Returns:
        X_windows: list of arrays, each shape (window_size, feature_dim)
        y_windows: list of labels
    """

    X_windows = []
    y_windows = []

    num_frames = sequence.shape[0]

    if num_frames < window_size:
        return X_windows, y_windows

    for start in range(0, num_frames - window_size + 1, stride):
        end = start + window_size
        window = sequence[start:end]

        X_windows.append(window)
        y_windows.append(label)

    return X_windows, y_windows

In [9]:
def build_lstm_dataset_from_df(split_df, window_size=25, stride=10):
    """
    Builds X and y arrays from video-level landmark files.

    Args:
        split_df: dataframe containing landmark_path and label
        window_size: number of frames per LSTM sample
        stride: overlap step

    Returns:
        X: numpy array of shape (num_windows, window_size, feature_dim)
        y: numpy array of shape (num_windows,)
        window_info: dataframe showing where each window came from
    """

    X = []
    y = []
    window_records = []

    for _, row in tqdm(split_df.iterrows(), total=len(split_df)):
        landmark_path = Path(row["landmark_path"])
        video_name = row["video_name"]
        label = int(row["label"])

        sequence = np.load(landmark_path)

        X_windows, y_windows = create_windows_from_sequence(
            sequence=sequence,
            label=label,
            window_size=window_size,
            stride=stride
        )

        for window_index, window in enumerate(X_windows):
            X.append(window)
            y.append(label)

            start_frame = window_index * stride
            end_frame = start_frame + window_size

            window_records.append({
                "video_name": video_name,
                "label": label,
                "window_index": window_index,
                "start_frame": start_frame,
                "end_frame": end_frame
            })

    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.float32)

    window_info = pd.DataFrame(window_records)

    return X, y, window_info

In [10]:
WINDOW_SIZE = 25
STRIDE = 10

X_train, y_train, train_window_info = build_lstm_dataset_from_df(
    train_df,
    window_size=WINDOW_SIZE,
    stride=STRIDE
)

X_val, y_val, val_window_info = build_lstm_dataset_from_df(
    val_df,
    window_size=WINDOW_SIZE,
    stride=STRIDE
)

X_test, y_test, test_window_info = build_lstm_dataset_from_df(
    test_df,
    window_size=WINDOW_SIZE,
    stride=STRIDE
)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

100%|██████████| 17/17 [00:00<00:00, 39.03it/s]

X_train shape: (837, 25, 1434)
y_train shape: (837,)
X_val shape: (138, 25, 1434)
y_val shape: (138,)
X_test shape: (183, 25, 1434)
y_test shape: (183,)


In [11]:
print("Train window labels:")
print(pd.Series(y_train).value_counts())

print("\nValidation window labels:")
print(pd.Series(y_val).value_counts())

print("\nTest window labels:")
print(pd.Series(y_test).value_counts())

Train window labels:
0.0    486
1.0    351
Name: count, dtype: int64

Validation window labels:
1.0    71
0.0    67
Name: count, dtype: int64

Test window labels:
0.0    106
1.0     77
Name: count, dtype: int64


In [12]:
TARGET_FPS = 5

def add_time_columns(window_info, target_fps=5):
    window_info = window_info.copy()
    window_info["start_time"] = window_info["start_frame"] / target_fps
    window_info["end_time"] = window_info["end_frame"] / target_fps
    return window_info

train_window_info = add_time_columns(train_window_info, TARGET_FPS)
val_window_info = add_time_columns(val_window_info, TARGET_FPS)
test_window_info = add_time_columns(test_window_info, TARGET_FPS)

train_window_info.head()

,video_name,label,window_index,start_frame,end_frame,start_time,end_time
0,trial_truth_057,0,0,0,25,0.0,5.0
1,trial_truth_057,0,1,10,35,2.0,7.0
2,trial_truth_057,0,2,20,45,4.0,9.0
3,trial_truth_057,0,3,30,55,6.0,11.0
4,trial_truth_057,0,4,40,65,8.0,13.0


In [13]:
np.save(PREPARED_OUTPUT_DIR / "X_train.npy", X_train)
np.save(PREPARED_OUTPUT_DIR / "y_train.npy", y_train)

np.save(PREPARED_OUTPUT_DIR / "X_val.npy", X_val)
np.save(PREPARED_OUTPUT_DIR / "y_val.npy", y_val)

np.save(PREPARED_OUTPUT_DIR / "X_test.npy", X_test)
np.save(PREPARED_OUTPUT_DIR / "y_test.npy", y_test)

train_window_info.to_csv(PREPARED_OUTPUT_DIR / "train_window_info.csv", index=False)
val_window_info.to_csv(PREPARED_OUTPUT_DIR / "val_window_info.csv", index=False)
test_window_info.to_csv(PREPARED_OUTPUT_DIR / "test_window_info.csv", index=False)

train_df.to_csv(PREPARED_OUTPUT_DIR / "train_videos.csv", index=False)
val_df.to_csv(PREPARED_OUTPUT_DIR / "val_videos.csv", index=False)
test_df.to_csv(PREPARED_OUTPUT_DIR / "test_videos.csv", index=False)

print("Prepared LSTM data saved to:", PREPARED_OUTPUT_DIR)

Prepared LSTM data saved to: prepared_lstm_data


In [14]:
import json

config = {
    "target_fps": TARGET_FPS,
    "window_size": WINDOW_SIZE,
    "stride": STRIDE,
    "feature_dim": int(X_train.shape[2]),
    "label_mapping": {
        "truthful": 0,
        "deceptive": 1
    },
    "split": {
        "train": len(train_df),
        "validation": len(val_df),
        "test": len(test_df)
    }
}

with open(PREPARED_OUTPUT_DIR / "preprocessing_config.json", "w") as f:
    json.dump(config, f, indent=4)

config

{'target_fps': 5,
 'window_size': 25,
 'stride': 10,
 'feature_dim': 1434,
 'label_mapping': {'truthful': 0, 'deceptive': 1},
 'split': {'train': 79, 'validation': 17, 'test': 17}}